# 06_interpretable_rules

This notebook describes **HH (High-High) multiplicity hotspots** with simple, interpretable rules.

**Goal**: For each HH component (connected region of high predictive variance), fit a shallow decision tree that distinguishes component members from the rest of the data. Report precision, recall, and the rule text. Rules are evaluated **out-of-sample** when component size permits.

**Inputs** (from spatial hotspot analysis):
- Rashomon predictions and variance
- LISA cluster labels → HH components
- Test feature space

## 1. Imports and paths

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'analysis').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from analysis.spatial import build_knn_graph, lisa_local, extract_hh_components
from analysis.rules import extract_component_rules, rules_summary_df

In [2]:
BASE_PATH = Path("../results/compas/seed=42_eps=0.01")
assert BASE_PATH.exists(), f"Result directory not found: {BASE_PATH.absolute()}"

## 2. Load artifacts and compute HH components

In [3]:
P = np.load(BASE_PATH / "P_test.npy")
metrics = np.load(BASE_PATH / "metrics.npz")
X_test = pd.read_csv(BASE_PATH / "X_test.csv")
v = metrics["variance"]

In [4]:
k = 10
W = build_knn_graph(X_test, k=k)
lisa_df = lisa_local(v, W)
comp_id, components = extract_hh_components(lisa_df, W, min_size=5)

print(f"Found {len(components)} HH components")
if len(components) == 0:
    print("No components with min_size=5. Try reducing min_size in extract_hh_components.")
else:
    for cid, inds in components.items():
        print(f"  Component {cid}: {len(inds)} observations")

Found 3 HH components
  Component 0: 10 observations
  Component 1: 6 observations
  Component 2: 11 observations


## 3. Extract interpretable rules for each component

In [5]:
if len(components) == 0:
    rules = {}
else:
    rules = extract_component_rules(
        X_test,
        components,
        max_depth=3,
        min_samples_leaf=5,
        test_size=0.3,
        seed=42,
    )

## 4. Summary table: precision and recall

In [6]:
summary = rules_summary_df(rules) if rules else pd.DataFrame()
summary

,component,n_hh,precision_train,recall_train,precision_eval,recall_eval,out_of_sample
0,0,10,0.368421,1.0,0.6,1.0,True
1,1,6,1.000000,1.0,NaN,NaN,False
2,2,11,1.000000,1.0,1.0,1.0,True


In [7]:
if len(summary) > 0:
    n_oos = summary["out_of_sample"].sum()
    print(f"Out-of-sample evaluated: {n_oos} of {len(summary)} components")
    if n_oos > 0:
        print(f"Mean precision (eval): {summary.loc[summary['out_of_sample'], 'precision_eval'].mean():.3f}")
        print(f"Mean recall (eval): {summary.loc[summary['out_of_sample'], 'recall_eval'].mean():.3f}")

Out-of-sample evaluated: 2 of 3 components
Mean precision (eval): 0.800
Mean recall (eval): 1.000


## 5. Rule text per component

In [8]:
for cid, r in rules.items():
    print(f"\n{'='*60}")
    print(f"Component {cid} (n={r['n_component']})")
    print(f"  Precision (train): {r['precision_train']:.3f}  Recall (train): {r['recall_train']:.3f}")
    if r['out_of_sample']:
        print(f"  Precision (eval):  {r['precision_eval']:.3f}  Recall (eval):  {r['recall_eval']:.3f}  [out-of-sample]")
    print(f"\n  Rule:")
    print(r['rule_text'])
if not rules:
    print("No rules to display.")


Component 0 (n=10)
  Precision (train): 0.368  Recall (train): 1.000
  Precision (eval):  0.600  Recall (eval):  1.000  [out-of-sample]

  Rule:
|--- age <= 20.50
|   |--- priors_count <= 0.50
|   |   |--- race_African-American <= 0.50
|   |   |   |--- class: 1
|   |   |--- race_African-American >  0.50
|   |   |   |--- class: 1
|   |--- priors_count >  0.50
|   |   |--- class: 0
|--- age >  20.50
|   |--- class: 0


Component 1 (n=6)
  Precision (train): 1.000  Recall (train): 1.000

  Rule:
|--- priors_count <= 22.50
|   |--- race_Native American <= 0.50
|   |   |--- class: 0
|   |--- race_Native American >  0.50
|   |   |--- class: 0
|--- priors_count >  22.50
|   |--- age <= 46.00
|   |   |--- class: 0
|   |--- age >  46.00
|   |   |--- class: 1


Component 2 (n=11)
  Precision (train): 1.000  Recall (train): 1.000
  Precision (eval):  1.000  Recall (eval):  1.000  [out-of-sample]

  Rule:
|--- age <= 20.50
|   |--- priors_count <= 0.50
|   |   |--- class: 0
|   |--- priors_count 

## 6. Interpretation

* **Precision**: Of all points the rule labels as "this HH region", how many truly belong to the HH component?
* **Recall**: Of all true HH points in this component, how many are captured by the rule?
* Rules are shallow decision trees (max_depth=3) for interpretability.
* When component size permits (≥10 HH points), rules are evaluated out-of-sample on a held-out 30% of the test set.